# Transformer 550M — self-resuming training (English-quality edition)

Target: **~554M params** (`target-550m`: d_model 1280, 26 layers, ctx 1024) — same size class as Qwen2.5-0.5B.

Quality English corpus: **Wikipedia (~4 GB) + Cosmopedia textbooks (~2 GB) + FineMath (~0.5 GB) + Python code (~1.5 GB)**.
Vocabulary: 32k BPE. Total ~40,000 steps (~2.6B tokens).

1. Runtime → Change runtime type → GPU (T4, free)
2. Run the cells top to bottom. Checkpoints + data live in Drive (`transformer-550m/`) and survive disconnects.

In [ ]:
#@title 1 · Setup (GPU check, repo, deps)
import torch, os, subprocess, sys, shutil, json, time
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('GPU:', gpu or 'NONE — enable it: Runtime > Change runtime type > GPU')
assert gpu, 'A GPU is required for the 550M run'
print('bf16 supported:', torch.cuda.is_bf16_supported())

REPO = '/content/transformer-llm'
IN_COLAB = 'google.colab' in sys.modules
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/samratbarman1013-commits/transformer-llm.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tokenizers', 'numpy', 'onnx', 'onnxruntime'], check=True)
os.chdir(REPO)
print('repo ready:', os.getcwd())

In [ ]:
#@title 2 · Mount Drive (checkpoint home) — Colab only
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/transformer-550m'
else:
    DRIVE = '/kaggle/working/transformer-550m'  # Kaggle: survives via Output dataset
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
STATUS = f'{DRIVE}/status.json'
print('checkpoint home:', DRIVE)

In [ ]:
#@title 3 - Corpus (high-quality English mix ~8 GB - skipped if already on Drive)
import subprocess, sys, os, json, gzip, urllib.request
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)

DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)
CORPUS = f'{DATA}/corpus.txt'
NL = chr(10)

if os.path.exists(f'{DRIVE}/bins/train.bin'):
    print('corpus already prepared on Drive - nothing to download')
elif os.path.exists(CORPUS) and os.path.getsize(CORPUS) > 5_000_000_000:
    print('corpus.txt present:', round(os.path.getsize(CORPUS)/1e9, 2), 'GB')
else:
    if os.path.exists(CORPUS):
        print('incomplete corpus found (', round(os.path.getsize(CORPUS)/1e9, 2), 'GB) - redownloading')
    with open(CORPUS, 'w', encoding='utf-8') as out:
        from datasets import load_dataset

        def stream_texts(ds, cap, label):
            n, last = 0, 0
            for ex in ds:
                t = (ex.get('text') or '').strip()
                if len(t) < 200:
                    t = ((ex.get('problem') or '') + ' ' + (ex.get('solution') or '')).strip()
                    if len(t) < 200:
                        continue
                out.write(t + NL); n += len(t) + 1
                if n >= cap:
                    break
                if n - last >= 500_000_000:
                    last = n; print(f'  {label}:', round(n/1e9, 2), 'GB downloaded')
            print(f'{label}:', round(n/1e9, 2), 'GB')

        # 1) English Wikipedia - encyclopedic reference text (~4 GB)
        print('downloading wikipedia (4 GB)... progress every 0.5 GB below')
        stream_texts(load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True),
                     4_000_000_000, 'wikipedia')
        # 2) Cosmopedia - synthetic textbooks/educational text, high quality (~2 GB)
        print('downloading cosmopedia (2 GB)... a few quiet minutes')
        stream_texts(load_dataset('HuggingFaceTB/cosmopedia', 'web_samples_v2', split='train', streaming=True),
                     2_000_000_000, 'cosmopedia')
        # 3) FineMath - curated math/reasoning text (~0.5 GB)
        print('downloading finemath (0.5 GB)... a few quiet minutes')
        stream_texts(load_dataset('HuggingFaceTB/finemath', 'finemath-4plus', split='train', streaming=True),
                     500_000_000, 'finemath')
        # 4) Python code (~1.5 GB) from codeparrot-clean
        cap, n = 1_500_000_000, 0
        for i, shard in enumerate(('file-000000000001.json.gz', 'file-000000000002.json.gz'), 1):
            print(f'downloading python code shard {i}/2 (~1 GB)... a few quiet minutes')
            path = f'{DATA}/{shard}'
            urllib.request.urlretrieve(
                f'https://huggingface.co/datasets/codeparrot/codeparrot-clean/resolve/main/{shard}', path)
            with gzip.open(path, 'rt', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    try:
                        ex = json.loads(line)
                    except Exception:
                        continue
                    c = ex.get('content') or ''
                    if len(c) < 200 or ex.get('autogenerated'):
                        continue
                    out.write(c + NL); n += len(c) + 1
                    if n >= cap:
                        break
            os.remove(path)
            if n >= cap:
                break
        print('python code:', round(n/1e9, 2), 'GB')
print('corpus ready')


In [ ]:
#@title 4 · Tokenizer (32k BPE) — trained once, reused across sessions
from transformer.tokenizer_bpe import train as train_tok
TOK = f'{DRIVE}/tokenizer.json'
if not (os.path.exists(TOK) and os.path.getsize(TOK) > 500_000):
    if os.path.exists(TOK): print('stub tokenizer found - retraining')
    train_tok(CORPUS, TOK, vocab_size=32768)
else:
    print('reusing tokenizer from Drive')

In [ ]:
#@title 5 · Pre-tokenize to train.bin / val.bin (~30 min once; cached to Drive)
from transformer.tokenizer_bpe import encode_file
import shutil, json
BINS = f'{DATA}/bins'
DBINS = f'{DRIVE}/bins'
if not os.path.exists(f'{BINS}/train.bin'):
    if os.path.exists(f'{DBINS}/train.bin'):
        shutil.copytree(DBINS, BINS, dirs_exist_ok=True)
        print('bins restored from Drive')
    else:
        encode_file(CORPUS, TOK, BINS, val_frac=0.001)
        shutil.copytree(BINS, DBINS, dirs_exist_ok=True)
        print('bins cached to Drive — future sessions skip the download entirely')
print(json.load(open(f'{BINS}/meta.json')))

In [ ]:
#@title 6 · TRAIN (auto-resume loop — re-run this cell after any disconnect)
TOTAL_STEPS = 40000        # ~2.6B tokens at 65k tokens/step
CHUNK_SECONDS = 2 * 3600   # checkpoint to Drive every 2 hours
CKPT = '/content/ckpt'     # fast local disk; mirrored to Drive each chunk

def latest_step():
    try: return json.load(open(STATUS)).get('step', 0)
    except Exception: return 0

n_gpu = torch.cuda.device_count()
while latest_step() < TOTAL_STEPS:
    base = ([sys.executable] if n_gpu == 1 else
            ['torchrun', f'--nproc_per_node={n_gpu}'])
    cmd = base + ['-m', 'transformer.train_v2',
        '--config', 'target-550m', '--data', BINS, '--out', CKPT,
        '--steps', str(TOTAL_STEPS), '--batch', '4', '--grad-accum', '16',
        '--ctx', '1024', '--lr', '2e-4', '--warmup', '400',
        '--eval-every', '500', '--grad-checkpoint', '--dropout', '0.0',
        '--max-seconds', str(CHUNK_SECONDS), '--status-out', STATUS]
    print('>> launching:', ' '.join(cmd))
    p = subprocess.run(cmd)
    for f in ('ckpt_last.pt', 'model_best.pt', 'train_log.jsonl'):
        src = f'{CKPT}/{f}'
        if os.path.exists(src): shutil.copy2(src, f'{DRIVE}/checkpoints/{f}')
    s = latest_step()
    print(f'chunk done (rc={p.returncode}) — step {s}/{TOTAL_STEPS}')
    if p.returncode != 0:
        time.sleep(30)  # transient GPU reset; loop retries
print('TRAINING COMPLETE — best model in', f'{DRIVE}/checkpoints/model_best.pt')

In [ ]:
#@title 7 - Chat with the trained 550M model
import torch
from transformer.config import CONFIGS
from transformer.model import GPT
from transformer.tokenizer_bpe import BpeTokenizer

ck = torch.load(f'{DRIVE}/checkpoints/model_best.pt', map_location='cpu', weights_only=False)
cfg = CONFIGS[ck['config']]
cfg.gradient_checkpointing = False
model = GPT(cfg, ck['vocab_size']); model.load_state_dict(ck['model']); model.eval()
tok = BpeTokenizer(f'{DRIVE}/tokenizer.json')
prompt = 'Once upon a time there was a little girl named Lily'
ids = torch.tensor([tok.encode(prompt)], dtype=torch.long)
out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_k=50)[0].tolist()
print(tok.decode(out[len(ids[0]):]))

## When training completes

- Export for the browser/phone: `python -m transformer.export_onnx --checkpoint <model_best.pt> --out web/model/model.onnx` (then re-deploy the site — see `TRAINING_500M.md`)
- Publish the weights: upload `model_best.pt` to the GitHub release (the supervisor can help)
- Next milestones: SFT on chat data (DPO later) — see `TRAINING_500M.md`